# Lesson 5b: n-Step Methods - Practical

**Reinforcement Learning Course - Powell Clark**

---

## Learning Objectives

By the end of this notebook, you will:

1. **Implement n-Step TD Prediction** - Configurable n-step bootstrapping
2. **Implement n-Step SARSA** - Multi-step on-policy control
3. **Implement TD(λ)** - Eligibility traces for prediction
4. **Implement SARSA(λ)** - Eligibility traces for control
5. **Compare All Variants** - Parameter sensitivity analysis
6. **Benchmark Performance** - Speed vs accuracy trade-offs

This notebook provides production-ready implementations of all n-step methods.

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, deque
import gymnasium as gym
from typing import Tuple, List, Dict
import time

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

np.random.seed(42)
print("✅ Setup complete")

## 1. n-Step TD Prediction

### Implementation

We implement n-step TD for evaluating a fixed policy.

In [ ]:
class NStepTD:
    """
    n-Step Temporal Difference Prediction.
    
    Estimates V(s) for a given policy using n-step returns:
    G_t^(n) = R_{t+1} + γR_{t+2} + ... + γ^{n-1}R_{t+n} + γ^n V(S_{t+n})
    """
    
    def __init__(self, n_states: int, n: int, alpha: float = 0.1, gamma: float = 0.99):
        """
        Args:
            n_states: Number of states
            n: Number of steps for bootstrapping
            alpha: Learning rate
            gamma: Discount factor
        """
        self.n_states = n_states
        self.n = n
        self.alpha = alpha
        self.gamma = gamma
        self.V = np.zeros(n_states)
        
    def update(self, episode: List[Tuple[int, float]]):
        """
        Update value function from complete episode.
        
        Args:
            episode: List of (state, reward) tuples
        """
        T = len(episode)
        
        # Process each time step
        for t in range(T):
            # Compute n-step return
            G = 0.0
            
            # Sum discounted rewards for n steps (or until episode end)
            for i in range(self.n):
                if t + i >= T:
                    break
                _, reward = episode[t + i]
                G += (self.gamma ** i) * reward
            
            # Add bootstrapped value if episode hasn't ended
            if t + self.n < T:
                next_state, _ = episode[t + self.n]
                G += (self.gamma ** self.n) * self.V[next_state]
            
            # Update value
            state, _ = episode[t]
            self.V[state] += self.alpha * (G - self.V[state])
    
    def evaluate(self, env, policy, n_episodes: int = 100):
        """
        Evaluate policy over multiple episodes.
        
        Args:
            env: Gymnasium environment
            policy: Policy function (state -> action)
            n_episodes: Number of episodes to run
        """
        for episode_num in range(n_episodes):
            episode = []
            state, _ = env.reset()
            
            # Generate episode
            for step in range(1000):
                action = policy(state)
                next_state, reward, terminated, truncated, _ = env.step(action)
                
                episode.append((state, reward))
                
                if terminated or truncated:
                    break
                    
                state = next_state
            
            # Update from complete episode
            self.update(episode)
        
        return self.V.copy()

print("✅ n-Step TD implemented")

### Test on Random Walk

We'll test n-step TD on a simple random walk to compare different values of n.

In [ ]:
class RandomWalk:
    """
    Random Walk environment for testing value prediction.
    
    States: 0, 1, 2, 3, 4, 5, 6
    - State 0: Terminal (left)
    - State 1-5: Non-terminal
    - State 6: Terminal (right, reward +1)
    
    Start: State 3
    Actions: Left or Right (equal probability)
    """
    
    def __init__(self, n_states: int = 7):
        self.n_states = n_states
        self.start_state = n_states // 2
        self.state = self.start_state
        
    def reset(self):
        self.state = self.start_state
        return self.state, {}
    
    def step(self, action):
        # Action: 0=left, 1=right
        if action == 0:
            self.state -= 1
        else:
            self.state += 1
        
        # Check terminal states
        if self.state == 0:
            return self.state, 0.0, True, False, {}
        elif self.state == self.n_states - 1:
            return self.state, 1.0, True, False, {}
        else:
            return self.state, 0.0, False, False, {}

# Random policy for testing
def random_policy(state):
    return np.random.choice([0, 1])

# True values (analytical solution for random walk)
true_values = np.array([0.0, 1/6, 2/6, 3/6, 4/6, 5/6, 1.0])

# Test different n values
n_values = [1, 2, 4, 8, 16]
results = {}

for n in n_values:
    env = RandomWalk()
    agent = NStepTD(n_states=7, n=n, alpha=0.1, gamma=1.0)
    V = agent.evaluate(env, random_policy, n_episodes=100)
    results[n] = V
    
    rmse = np.sqrt(np.mean((V - true_values) ** 2))
    print(f"n={n:2d}: RMSE = {rmse:.4f}")

print("\n✅ Random Walk test complete")

In [ ]:
# Visualize results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot learned values vs true values
states = np.arange(7)
ax1.plot(states, true_values, 'k--', linewidth=2, label='True Values', marker='o')
for n, V in results.items():
    ax1.plot(states, V, marker='s', label=f'n={n}')
ax1.set_xlabel('State')
ax1.set_ylabel('Value')
ax1.set_title('n-Step TD: Learned vs True Values')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot errors
for n, V in results.items():
    errors = np.abs(V - true_values)
    ax2.plot(states, errors, marker='s', label=f'n={n}')
ax2.set_xlabel('State')
ax2.set_ylabel('Absolute Error')
ax2.set_title('n-Step TD: Prediction Errors')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. n-Step SARSA

### Implementation

n-Step SARSA for control: learns Q(s,a) using n-step returns.

In [ ]:
class NStepSARSA:
    """
    n-Step SARSA Control Algorithm.
    
    Learns optimal policy using n-step action-value returns:
    G_t^(n) = R_{t+1} + γR_{t+2} + ... + γ^{n-1}R_{t+n} + γ^n Q(S_{t+n}, A_{t+n})
    """
    
    def __init__(self, n_actions: int, n: int = 3, alpha: float = 0.1, 
                 gamma: float = 0.99, epsilon: float = 0.1):
        """
        Args:
            n_actions: Number of actions
            n: Number of steps for bootstrapping
            alpha: Learning rate
            gamma: Discount factor
            epsilon: Exploration rate for ε-greedy policy
        """
        self.n_actions = n_actions
        self.n = n
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.Q = defaultdict(lambda: np.zeros(n_actions))
        
    def get_action(self, state) -> int:
        """ε-greedy action selection."""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        else:
            return np.argmax(self.Q[state])
    
    def train_episode(self, env) -> float:
        """
        Train on one episode using n-step SARSA.
        
        Returns:
            Total episode reward
        """
        # Storage for trajectory
        states = []
        actions = []
        rewards = [0]  # R_0 is unused, start from R_1
        
        # Initialize
        state, _ = env.reset()
        action = self.get_action(state)
        
        states.append(state)
        actions.append(action)
        
        T = float('inf')
        t = 0
        total_reward = 0
        
        while True:
            if t < T:
                # Take action
                next_state, reward, terminated, truncated, _ = env.step(action)
                total_reward += reward
                
                states.append(next_state)
                rewards.append(reward)
                
                if terminated or truncated:
                    T = t + 1
                else:
                    action = self.get_action(next_state)
                    actions.append(action)
            
            # τ is the time whose estimate is being updated
            tau = t - self.n + 1
            
            if tau >= 0:
                # Compute n-step return
                G = 0.0
                for i in range(tau + 1, min(tau + self.n, T) + 1):
                    G += (self.gamma ** (i - tau - 1)) * rewards[i]
                
                # Add bootstrapped value if not at terminal
                if tau + self.n < T:
                    G += (self.gamma ** self.n) * self.Q[states[tau + self.n]][actions[tau + self.n]]
                
                # Update Q-value
                s, a = states[tau], actions[tau]
                self.Q[s][a] += self.alpha * (G - self.Q[s][a])
            
            if tau == T - 1:
                break
                
            t += 1
        
        return total_reward
    
    def train(self, env, n_episodes: int = 500, verbose: bool = True) -> List[float]:
        """
        Train agent over multiple episodes.
        
        Returns:
            List of episode rewards
        """
        rewards = []
        
        for episode in range(n_episodes):
            ep_reward = self.train_episode(env)
            rewards.append(ep_reward)
            
            if verbose and (episode + 1) % 100 == 0:
                avg_reward = np.mean(rewards[-100:])
                print(f"Episode {episode + 1}/{n_episodes}, Avg Reward (last 100): {avg_reward:.2f}")
        
        return rewards
    
    def get_policy(self):
        """Extract greedy policy from Q-values."""
        policy = {}
        for state in self.Q.keys():
            policy[state] = np.argmax(self.Q[state])
        return policy

print("✅ n-Step SARSA implemented")

### Test on CliffWalking

CliffWalking is a classic gridworld problem ideal for testing n-step methods.

In [ ]:
# Compare different n values on CliffWalking
env = gym.make('CliffWalking-v0')
n_values = [1, 3, 5, 10]
results_nstep = {}

print("Training n-Step SARSA with different n values...\n")

for n in n_values:
    print(f"\n{'='*50}")
    print(f"Training with n={n}")
    print('='*50)
    
    agent = NStepSARSA(n_actions=env.action_space.n, n=n, alpha=0.1, 
                       gamma=1.0, epsilon=0.1)
    rewards = agent.train(env, n_episodes=500, verbose=True)
    results_nstep[n] = rewards

print("\n✅ CliffWalking comparison complete")

In [ ]:
# Visualize learning curves
fig, ax = plt.subplots(figsize=(12, 6))

for n, rewards in results_nstep.items():
    # Smooth with moving average
    window = 50
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, label=f'n={n}', linewidth=2)

ax.set_xlabel('Episode')
ax.set_ylabel('Reward (50-episode moving average)')
ax.set_title('n-Step SARSA: Learning Curves on CliffWalking')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print final performance
print("\nFinal Performance (last 100 episodes):")
for n, rewards in results_nstep.items():
    final_perf = np.mean(rewards[-100:])
    print(f"n={n:2d}: {final_perf:7.2f}")

## 3. TD(λ) with Eligibility Traces

### Implementation

TD(λ) uses eligibility traces for efficient online learning.

In [ ]:
class TDLambda:
    """
    TD(λ) with Eligibility Traces for Prediction.
    
    Online algorithm equivalent to forward-view λ-return:
    G_t^λ = (1-λ) Σ λ^{n-1} G_t^(n)
    """
    
    def __init__(self, n_states: int, lambda_: float = 0.9, alpha: float = 0.1, 
                 gamma: float = 0.99, trace_type: str = 'accumulating'):
        """
        Args:
            n_states: Number of states
            lambda_: Trace decay parameter (0 = TD(0), 1 = MC)
            alpha: Learning rate
            gamma: Discount factor
            trace_type: 'accumulating' or 'replacing' traces
        """
        self.n_states = n_states
        self.lambda_ = lambda_
        self.alpha = alpha
        self.gamma = gamma
        self.trace_type = trace_type
        self.V = np.zeros(n_states)
        
    def train_episode(self, env, policy) -> Tuple[float, np.ndarray]:
        """
        Train on one episode using TD(λ).
        
        Args:
            env: Environment
            policy: Policy function (state -> action)
        
        Returns:
            Tuple of (total_reward, max_trace) for analysis
        """
        # Initialize eligibility traces
        z = np.zeros(self.n_states)
        
        state, _ = env.reset()
        total_reward = 0
        max_trace = 0
        
        for step in range(1000):
            action = policy(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            
            # Compute TD error
            delta = reward + self.gamma * self.V[next_state] * (not terminated) - self.V[state]
            
            # Update eligibility trace for current state
            if self.trace_type == 'accumulating':
                z[state] += 1
            else:  # replacing
                z[state] = 1
            
            # Update all state values proportional to their traces
            self.V += self.alpha * delta * z
            
            # Decay all traces
            z *= self.gamma * self.lambda_
            
            max_trace = max(max_trace, np.max(z))
            
            if terminated or truncated:
                break
                
            state = next_state
        
        return total_reward, max_trace
    
    def evaluate(self, env, policy, n_episodes: int = 100):
        """
        Evaluate policy over multiple episodes.
        """
        for episode in range(n_episodes):
            self.train_episode(env, policy)
        
        return self.V.copy()

print("✅ TD(λ) implemented")

### Compare Different λ Values

In [ ]:
# Test TD(λ) with different λ values on Random Walk
lambda_values = [0.0, 0.5, 0.9, 0.95, 0.99, 1.0]
results_lambda = {}

print("Testing TD(λ) with different λ values...\n")

for lam in lambda_values:
    env = RandomWalk()
    agent = TDLambda(n_states=7, lambda_=lam, alpha=0.1, gamma=1.0)
    V = agent.evaluate(env, random_policy, n_episodes=100)
    results_lambda[lam] = V
    
    rmse = np.sqrt(np.mean((V - true_values) ** 2))
    print(f"λ={lam:.2f}: RMSE = {rmse:.4f}")

print("\n✅ λ comparison complete")

In [ ]:
# Visualize TD(λ) results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot learned values
states = np.arange(7)
ax1.plot(states, true_values, 'k--', linewidth=2, label='True Values', marker='o')
for lam, V in results_lambda.items():
    ax1.plot(states, V, marker='s', label=f'λ={lam}')
ax1.set_xlabel('State')
ax1.set_ylabel('Value')
ax1.set_title('TD(λ): Learned vs True Values')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot RMSE vs λ
rmses = [np.sqrt(np.mean((results_lambda[lam] - true_values) ** 2)) for lam in lambda_values]
ax2.plot(lambda_values, rmses, 'o-', linewidth=2, markersize=8)
ax2.set_xlabel('λ')
ax2.set_ylabel('RMSE')
ax2.set_title('TD(λ): Error vs Lambda')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. SARSA(λ)

### Implementation

SARSA(λ) extends eligibility traces to control.

In [ ]:
class SARSALambda:
    """
    SARSA(λ) with Eligibility Traces for Control.
    
    On-policy control using eligibility traces:
    - Maintains trace z(s,a) for each state-action pair
    - Updates all Q(s,a) proportional to their traces
    """
    
    def __init__(self, n_actions: int, lambda_: float = 0.9, alpha: float = 0.1,
                 gamma: float = 0.99, epsilon: float = 0.1, trace_type: str = 'accumulating'):
        """
        Args:
            n_actions: Number of actions
            lambda_: Trace decay parameter
            alpha: Learning rate
            gamma: Discount factor
            epsilon: Exploration rate
            trace_type: 'accumulating' or 'replacing'
        """
        self.n_actions = n_actions
        self.lambda_ = lambda_
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.trace_type = trace_type
        self.Q = defaultdict(lambda: np.zeros(n_actions))
        
    def get_action(self, state) -> int:
        """ε-greedy action selection."""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        else:
            return np.argmax(self.Q[state])
    
    def train_episode(self, env) -> float:
        """
        Train on one episode using SARSA(λ).
        
        Returns:
            Total episode reward
        """
        # Initialize eligibility traces (sparse storage)
        z = defaultdict(lambda: np.zeros(self.n_actions))
        
        state, _ = env.reset()
        action = self.get_action(state)
        total_reward = 0
        
        for step in range(1000):
            # Take action
            next_state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            next_action = self.get_action(next_state)
            
            # Compute TD error
            delta = reward + self.gamma * self.Q[next_state][next_action] * (not terminated) - self.Q[state][action]
            
            # Update eligibility trace for current state-action
            if self.trace_type == 'accumulating':
                z[state][action] += 1
            else:  # replacing
                z[state][action] = 1
            
            # Update all Q-values and decay traces
            states_to_update = list(z.keys())
            for s in states_to_update:
                self.Q[s] += self.alpha * delta * z[s]
                z[s] *= self.gamma * self.lambda_
                
                # Remove near-zero traces for efficiency
                if np.max(np.abs(z[s])) < 1e-8:
                    del z[s]
            
            if terminated or truncated:
                break
                
            state = next_state
            action = next_action
        
        return total_reward
    
    def train(self, env, n_episodes: int = 500, verbose: bool = True) -> List[float]:
        """
        Train agent over multiple episodes.
        """
        rewards = []
        
        for episode in range(n_episodes):
            ep_reward = self.train_episode(env)
            rewards.append(ep_reward)
            
            if verbose and (episode + 1) % 100 == 0:
                avg_reward = np.mean(rewards[-100:])
                print(f"Episode {episode + 1}/{n_episodes}, Avg Reward (last 100): {avg_reward:.2f}")
        
        return rewards
    
    def get_policy(self):
        """Extract greedy policy."""
        policy = {}
        for state in self.Q.keys():
            policy[state] = np.argmax(self.Q[state])
        return policy

print("✅ SARSA(λ) implemented")

### Test on CliffWalking

In [ ]:
# Compare SARSA(λ) with different λ values
env = gym.make('CliffWalking-v0')
lambda_values = [0.0, 0.5, 0.9, 0.99]
results_sarsa_lambda = {}

print("Training SARSA(λ) with different λ values...\n")

for lam in lambda_values:
    print(f"\n{'='*50}")
    print(f"Training with λ={lam}")
    print('='*50)
    
    agent = SARSALambda(n_actions=env.action_space.n, lambda_=lam, 
                        alpha=0.1, gamma=1.0, epsilon=0.1)
    rewards = agent.train(env, n_episodes=500, verbose=True)
    results_sarsa_lambda[lam] = rewards

print("\n✅ SARSA(λ) comparison complete")

In [ ]:
# Visualize SARSA(λ) learning curves
fig, ax = plt.subplots(figsize=(12, 6))

for lam, rewards in results_sarsa_lambda.items():
    window = 50
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, label=f'λ={lam}', linewidth=2)

ax.set_xlabel('Episode')
ax.set_ylabel('Reward (50-episode moving average)')
ax.set_title('SARSA(λ): Learning Curves on CliffWalking')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print final performance
print("\nFinal Performance (last 100 episodes):")
for lam, rewards in results_sarsa_lambda.items():
    final_perf = np.mean(rewards[-100:])
    print(f"λ={lam:.2f}: {final_perf:7.2f}")

## Summary and Key Insights

### Implementations Complete

✅ **n-Step TD Prediction** - Multi-step bootstrapping for value estimation  
✅ **n-Step SARSA** - Multi-step on-policy control  
✅ **TD(λ)** - Eligibility traces for prediction  
✅ **SARSA(λ)** - Eligibility traces for control  

### Key Findings

**1. Optimal n:**
- Small problems: n=3-5 often best
- Large problems: n=1-3 preferred
- Trade-off: bias (low n) vs variance (high n)

**2. Optimal λ:**
- λ=0.9-0.95 works well in most cases
- λ=0: Equivalent to 1-step TD
- λ=1: Approaches Monte Carlo

**3. n-Step vs TD(λ):**
- TD(λ) often faster convergence
- n-Step simpler to implement
- TD(λ) requires O(|S|) updates per step
- Sparse traces make TD(λ) practical

**4. Trace Types:**
- Accumulating: Better for revisited states
- Replacing: Prevents trace overflow
- Similar performance in most cases

### Practical Recommendations

**Start with:**
- SARSA(λ) with λ=0.9
- Or 3-step SARSA

**Tune based on:**
- Episode length (longer → smaller n/λ)
- Stochasticity (higher → smaller n/λ)
- Computational budget (limited → smaller n, avoid traces)

---

**Lesson 5b Complete!** 🎉

You now have production-ready implementations of all n-step methods and eligibility traces, completing the tabular RL toolkit.